<a href="https://colab.research.google.com/github/ywchanna2001/Protein_Conformational_Ensembles_Prediction/blob/dev/Protein_Conformational_Ensemble_Prediction_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mounting google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import urllib.request

def fetch_and_print_sequence(pdb_id, chain_id, target_name):
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id.upper()}"
    try:
        with urllib.request.urlopen(url) as response:
            fasta_data = response.read().decode('utf-8')

        records = fasta_data.split('>')
        for record in records:
            if not record.strip():
                continue
            header, *seq_lines = record.strip().split('\n')
            # Extract sequence matching target chain
            if f"Chain {chain_id.upper()}" in header or f"Chains {chain_id.upper()}" in header or f"_{chain_id.upper()}" in header:
                sequence = "".join(seq_lines).replace(" ", "")
                print(f"### {target_name} ({pdb_id.upper()}_{chain_id.upper()}) - {len(sequence)} aa")
                print(f">{pdb_id.upper()}_{chain_id.upper()}\n{sequence}\n")
                return
        # Fallback to first record
        if len(records) > 1:
            header, *seq_lines = records[1].strip().split('\n')
            sequence = "".join(seq_lines).replace(" ", "")
            print(f"### {target_name} ({pdb_id.upper()}_{chain_id.upper()}) - Fallback - {len(sequence)} aa")
            print(f">{pdb_id.upper()}_{chain_id.upper()}\n{sequence}\n")
    except Exception as e:
        print(f"Error fetching {target_name}: {e}")

targets = [
    ("4ake", "A", "1. Adenylate Kinase (AdK)"),
    ("1gqn", "A", "2. DHQase"),
    ("6nc7", "A", "3. Lipid II Flippase"),
    ("1urp", "D", "4. Ribose-Binding Protein"),
    ("2qke", "A", "5. KaiB")
]

print("── Fetching Amino Acid Sequences from RCSB PDB ────────────────\n")
for pdb, chain, name in targets:
    fetch_and_print_sequence(pdb, chain, name)

In [ ]:
#@title Input protein sequence(s)
from google.colab import files
import os
import re
import hashlib
import random

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

query_sequence = 'MRIILLGAPGAGKGTQAQFIMEKYGIPQISTGDMLRAAVKSGSELGKQAKDIMDAGKLVT DELVIALVKERIAQEDCRNGFLLDGFPRTIPQADAMKEAGINVDYVLEFDVPDELIVDRI VGRRVHAPSGRVYHVKFNPPKVEGKDDVTGEELTTRKDDQEETVRKRLVEYHQMTAPLIG YYSKEAEAGNTKYAKVDGTKPVAEVRADLEKILG' #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
jobname = 'Test1_by_channa' #@param {type:"string"}
# number of models to use
num_relax = 5 #@param [0, 1, 5] {type:"raw"}
#@markdown - specify how many of the top ranked structures to relax using amber
template_mode = "none" #@param ["none", "pdb100","custom"]
#@markdown - `none` = no template information is used. `pdb100` = detect templates in pdb100 (see [notes](#pdb100)). `custom` - upload and search own templates (PDB or mmCIF format, see [notes](#custom_templates))

use_amber = num_relax > 0

# remove whitespaces
query_sequence = "".join(query_sequence.split())

basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# check if directory with jobname exists.
# If folder exist return 'False'
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

# make directory to save results
os.makedirs(jobname, exist_ok=True)

# save queries
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
  text_file.write(f"id,sequence\n{jobname},{query_sequence}")

if template_mode == "pdb100":
  use_templates = True
  custom_template_path = None
elif template_mode == "custom":
  custom_template_path = os.path.join(jobname,f"template")
  os.makedirs(custom_template_path, exist_ok=True)
  uploaded = files.upload()
  use_templates = True
  for fn in uploaded.keys():
    os.rename(fn,os.path.join(custom_template_path,fn))
else:
  custom_template_path = None
  use_templates = False

print("jobname",jobname)
print("sequence",query_sequence)
print("length",len(query_sequence.replace(":","")))

In [ ]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@title MSA Options
#@markdown ### MSA options (custom MSA upload, single sequence, pairing mode)
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref","single_sequence","custom"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired","paired","unpaired"] {type:"string"}
#@markdown - "unpaired_paired" = pair sequences from same species + unpaired MSA, "unpaired" = seperate MSA for each chain, "paired" - only use paired sequences.

# decide which a3m to use
if "mmseqs2" in msa_mode:
  a3m_file = os.path.join(jobname,f"{jobname}.a3m")

# we don't use these options(custom, single_sequence) for this research
elif msa_mode == "custom":
  a3m_file = os.path.join(jobname,f"{jobname}.custom.a3m")
  if not os.path.isfile(a3m_file):
    custom_msa_dict = files.upload()
    custom_msa = list(custom_msa_dict.keys())[0]
    header = 0
    import fileinput
    for line in fileinput.FileInput(custom_msa,inplace=1):
      if line.startswith(">"):
         header = header + 1
      if not line.rstrip():
        continue
      if line.startswith(">") == False and header == 1:
         query_sequence = line.rstrip()
      print(line, end='')

    os.rename(custom_msa, a3m_file)
    queries_path=a3m_file
    print(f"moving {custom_msa} to {a3m_file}")

else:
  a3m_file = os.path.join(jobname,f"{jobname}.single_sequence.a3m")
  with open(a3m_file, "w") as text_file:
    text_file.write(">1\n%s" % query_sequence)

# MODULE 1:  MSA GENERATION

In [ ]:
#@title MSA GENERATION
import time
import colabfold
from colabfold.colabfold import run_mmseqs2
import colabfold.colabfold as cf_module

# In this research we always use mmseques2_uniref_env as the msa option
search_mode = msa_mode
use_env = "env" in search_mode

print(f"Target Sequence: {query_sequence[:30]}...")
print("full target sequence: ", query_sequence)
print(f"Job Name: {jobname}")

# Delete any stale cached .a3m file so the API is actually called
a3m_cache = os.path.join(jobname, f"{jobname}.a3m")
if os.path.isfile(a3m_cache):
    os.remove(a3m_cache)
    print("Removed cached .a3m — forcing fresh API search...")

print("Status: Contacting MMseqs2 server (this can take 1–3 minutes)...")

try:
    #run_mmseqs2 returns a LIST of a3m content strings

    result = run_mmseqs2(
        x=query_sequence,
        prefix=jobname,
        use_env=use_env,
        use_filter=True,           # Required for proper hit filtering. This removes redundant and low quality hits
        use_pairing=False,
        user_agent= "qevagna@gmail.com"
    )

    print("result: ",result)

    # Flatten the result into individual lines correctly
    # result is a list of strings; each string can be a full a3m block
    if isinstance(result, list):
        print('result is a list')
        a3m_lines = "\n".join(result).splitlines(keepends=True)
    else:
        a3m_lines = result.splitlines(keepends=True)

    print("a3m_lines: ", a3m_lines)

    # Count sequences properly (lines starting with '>')
    num_seqs = sum(1 for line in a3m_lines if line.startswith(">"))

    print(f"Total lines in result: {len(a3m_lines)}")
    print(f"Sequences found (headers): {num_seqs}")

    if num_seqs <= 1:
        print("❌ ERROR: Server returned 0 hits.")
        print("Debugging info:")
        print("".join(a3m_lines[:10]))  # Show first 10 lines for diagnosis
        print("\nPossible causes:")
        print("  - MMseqs2 API server is down or overloaded (try again in 1 min)")
        print("  - Sequence too short/non-standard for database hits")
        print("  - Network issue in Colab (try Runtime > Restart session and run all)")
    else:
        # Save valid MSA
        msa_module_dir = "/content/drive/MyDrive/Research/MSA_output"
        os.makedirs(msa_module_dir, exist_ok=True)
        msa_file_path = os.path.join(msa_module_dir, f"{jobname}.a3m")

        with open(msa_file_path, "w") as f:
            f.writelines(a3m_lines)

        print(f"✅ SUCCESS: {num_seqs} sequences saved to {msa_file_path}")

except Exception as e:
    print(f"❌ SERVER ERROR: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def plot_msa(feature_dict, sort_lines=True, dpi=100):
    import matplotlib.pyplot as plt
    import numpy as np

    # Extract data from  feature_dict
    msa = feature_dict["msa"]
    # 21 is the standard GAP_IDX
    non_gaps = (msa != 21).astype(float)
    non_gaps[non_gaps == 0] = np.nan

    # Calculate identity if not pre-calculated
    seq = msa[0]
    # Simple identity calculation
    identity = np.nanmean(msa == seq, axis=1)
    # print("\nidentity :",identity)

    # Prepare lines for plotting (identity * non_gap_mask)
    lines = non_gaps * identity[:, None]
    # print("\nlines before  sort:",lines)

    # Sort lines by identity
    lines = lines[np.argsort(identity)]
    # print("\nlines after sort:",lines)

    # 4. Rendering
    plt.figure(figsize=(10, 8), dpi=dpi)
    plt.title("Sequence coverage")

    # Plotting the matrix
    plt.imshow(lines,
              interpolation='nearest', aspect='auto',
              cmap="rainbow_r", vmin=0, vmax=1, origin='lower',
              extent=(0, lines.shape[1], 0, lines.shape[0]))

    # Add coverage curve (count non-NaN values per column)
    coverage = np.nansum(~np.isnan(lines), axis=0)
    plt.plot(coverage, color='black', lw=1.5)

    plt.xlim(0, lines.shape[1])
    plt.ylim(0, lines.shape[0])
    plt.colorbar(label="Sequence identity to query")
    plt.xlabel("Positions")
    plt.ylabel("Sequences")
    plt.show()

In [ ]:
#@title Sequence Coverage Plot
# Build and Plot
import numpy as np
import matplotlib.pyplot as plt

# 1. Parse A3M data
sequences = []
current_seq =[]
for line in a3m_lines:
    line = line.strip()
    if not line or line.startswith("#"): continue
    if line.startswith(">"):
        if current_seq: sequences.append("".join(current_seq))
        current_seq =[]
    else:
        current_seq.append("".join(c if (c.isupper() or c == "-") else "" for c in line))
if current_seq: sequences.append("".join(current_seq))

# print("current_seq: ", current_seq)
# print("sequences: ", sequences)

# 2. Build the Matrix
query_len = len(sequences[0])
print("query_length: ",query_len)

GAP_IDX = 21 #  lowercase letters and gaps(-)
res_map = {c: i for i, c in enumerate("ARNDCQEGHILKMFPSTWYV")}
# print("\nres_map: ", res_map)

msa_list = [[res_map.get(c.upper(), GAP_IDX) for c in seq[:query_len]] for seq in sequences]
# print("\nmsa_list: ",msa_list)
# Pad shorter sequences
msa_arr = np.array([row + [GAP_IDX]*(query_len-len(row)) for row in msa_list], dtype=np.int32)
# print("\nmsa_arr shape: ",msa_arr.shape)
# print("msa_arr: ",msa_arr)

# 3. Create the global input_features dictionary
input_features = {
    "msa": msa_arr,
    "num_alignments": np.sum(msa_arr != GAP_IDX, axis=0).astype(np.int32)
}

# print("\ninput features: ", input_features)

print(f"✅ 'input_features' created. Shape: {input_features['msa'].shape}")

# 4. Run the plotting function
print("Plotting sequence coverage...")
plot_msa(input_features)

# MODULE 2: DBSCAN CLUSTERER
Official DBSCAN logic from AFCluster

In [ ]:
import numpy as np
from Bio import SeqIO

#This function calculates the "most common" residue at every single position
#in a set of sequences to generate a single representative sequence.
def consensusVoting(seqs):
    consensus = ""
    residues = "ARNDCQEGHILKMFPSTWYV-"
    n_chars = len(seqs[0])
    for i in range(n_chars):
        baseArray = [x[i] for x in seqs]
        baseCount = np.array([baseArray.count(a) for a in list(residues)])
        vote = np.argmax(baseCount)
        consensus += residues[vote]
    return consensus

def encode_seqs(seqs, max_len=108, alphabet="ARNDCQEGHILKMFPSTWYV-"):
    # Official OHE logic: Creates a 3D matrix (N, L, 21) then flattens
    arr = np.zeros([len(seqs), max_len, len(alphabet)])
    for j, seq in enumerate(seqs):
        for i, char in enumerate(seq):
            if char in alphabet:
                arr[j, i, alphabet.index(char)] += 1
    return arr.reshape([len(seqs), max_len * len(alphabet)])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import json


# 1. Load Data
with open(f"/content/drive/MyDrive/Research/MSA_output/{jobname}.a3m", "r") as f:
    lines = f.readlines()
# print("lines: ",lines)

# Parse sequences — remove lowercase insertions, keep uppercase + gaps
all_seqs = [
    ''.join([c for c in line.strip() if c.isupper() or c == '-'])
    for line in lines
    if line.strip() and not line.startswith('>')
]
# print("\nall_seqs: ",all_seqs)


L = len(all_seqs[0])
print("L(length of the 1st sequence): ", L)

# 2. Separate query from homologues (AFCluster logic)
# Query is always the first sequence in the .a3m file
# DBSCAN runs on homologues only — query is added back to each cluster later
query_seq  = all_seqs[0]
homo_seqs  = all_seqs[1:]
# print("query_seq: ",query_seq)
# print("homo_seqs: ",homo_seqs)

# Build DataFrame of homologues
df = pd.DataFrame({'sequence': homo_seqs})
# print("\ndf before frac_gaps\n: ",df)
df['frac_gaps'] = [x.count('-') / L for x in df['sequence']]
# print("df shape: ",df.shape)
# print("df: \n",df)

# 3. Gap filtering (AFCluster's 0.25 cutoff logic)
before = len(df)
df = df.loc[df.frac_gaps < 0.25].reset_index(drop=True)
print(f"✅ Gap filter: {before - len(df)} sequences removed, "
      f"{len(df)} remaining")

# 4. One-Hot Encoding (AFCluster exact logic)
ohe_seqs = encode_seqs(df.sequence.tolist(), max_len=L)
print(f"✅ OHE matrix shape: {ohe_seqs.shape}")
# print("\nohe_seqs: ", ohe_seqs)


# 5. Eps scanning (AFCluster logic — scan on 25% subsample)
# Scans eps from 3.0 to 20.0 in steps of 0.5
# Selects eps that maximises number of clusters
subsample = df.sample(frac=0.25, random_state=42)
# print("\nsubsample: ",subsample)
testset = encode_seqs(subsample.sequence.tolist(), max_len=L)
# print("\ntestset: ",testset)

print("\nScanning eps values on 25% subsample...")
print(f"{'eps':>6}  {'n_clusters':>10}  {'n_noise':>8}")

eps_values  = np.arange(3.0, 20.5, 0.5)
n_clusters_per_eps = []



for eps in eps_values:
    clustering = DBSCAN(eps=eps, min_samples=3).fit(testset)             #<== DBSCAN algorithm execution
    labels     = clustering.labels_
    n_clust    = len(set(labels)) - (1 if -1 in labels else 0)           #<== removing noise
    n_noise    = list(labels).count(-1)
    print(f"{eps:>6.1f}  {n_clust:>10}  {n_noise:>8}")
    n_clusters_per_eps.append(n_clust)
    # Early stop — AFCluster stops if n_clusters==1 after eps>10
    if eps > 10 and n_clust == 1:
        break

eps_to_use = eps_values[np.argmax(n_clusters_per_eps)]
print("\nn_clusters_per_eps: ",n_clusters_per_eps)
print(f"\n✅ Selected eps = {eps_to_use:.2f}  "
      f"(maximises n_clusters = {max(n_clusters_per_eps)})")

# 6. Final DBSCAN on full dataset
db = DBSCAN(eps=eps_to_use, min_samples=3).fit(ohe_seqs)
df['dbscan_label'] = db.labels_

# 7. Set the three variables Module 3 depends on
# These MUST be set here so Module 3 (Cell 10) works correctly.
cluster_labels = db.labels_                                          # ndarray
n_clusters     = len([x for x in df.dbscan_label.unique() if x >= 0])


# lines is already set above from f.readlines()
# Verify all three are set
print(f"\n✅ cluster_labels : shape {cluster_labels.shape}, "
      f"dtype {cluster_labels.dtype}")
print(f"✅ n_clusters     : {n_clusters}")
print(f"✅ lines          : {len(lines)} raw lines from .a3m file")

# 8. Summary
n_noise = list(cluster_labels).count(-1)
print(f"\n── Clustering summary ──────────────────────────────────────")
print(f"   Total homologue sequences : {len(df)}")
print(f"   Clusters found            : {n_clusters}")
print(f"   Noise (unclustered)       : {n_noise} "
      f"({n_noise/len(df)*100:.1f}%)")
for cid in sorted([x for x in df.dbscan_label.unique() if x >= 0]):
    size = len(df[df.dbscan_label == cid])
    print(f"   Cluster {cid:>3}              : {size} sequences")

In [ ]:
#@title SUB-MSA GENERATION & STORAGE
import os
import shutil

# 1. Create a directory to store the sub-MSAs
clusters_dir = "DBSCAN_outputs"

if os.path.exists(clusters_dir):
    print(f"Clearing previous sub-MSAs from '{clusters_dir}' to prevent contamination...")
    shutil.rmtree(clusters_dir)  # Wipes the entire directory

os.makedirs(clusters_dir, exist_ok=True)

if 'cluster_labels' in locals() and 'lines' in locals():
    print(f"Generating sub-MSAs for {n_clusters} clusters...")

    # 2. Group the original A3M text by clusters
    # Headers start with '>', sequences follow.
    msa_data = []
    current_header = ""
    for line in lines:
        if line.startswith(">"):
            current_header = line.strip()
        else:
            if current_header:
                msa_data.append((current_header, line.strip()))
                current_header = ""

    # 3. Create a file for each cluster
    for cluster_id in range(n_clusters):
        cluster_file = os.path.join(clusters_dir, f"cluster_{cluster_id}.a3m")

        # Get indices of sequences belonging to this cluster
        indices = [i for i, label in enumerate(cluster_labels) if label == cluster_id]

        with open(cluster_file, "w") as f:
            # AlphaFold ALWAYS needs the query sequence (index 0) at the top of every cluster
            f.write(f"{msa_data[0][0]}\n{msa_data[0][1]}\n")

            # Write the rest of the sequences in the cluster
            for idx in indices:
                if idx == 0: continue # Skip query if it's already in the cluster
                header, seq = msa_data[idx]
                f.write(f"{header}\n{seq}\n")

        print(f"✅ Saved Cluster {cluster_id} ({len(indices)} sequences) to {cluster_file}")

    print(f"\n--- ALL SUB-MSAs STORED IN '{clusters_dir}' FOLDER ---")
else:
    print("❌ Error: Please run Module 2 (Clustering) first to generate labels.")

In [ ]:
import seaborn as sns

# Perform PCA for visualization
pca = PCA(n_components=2)
# print("pca: ",pca)
coords = pca.fit_transform(ohe_seqs)
# print("coords :,coords")
df['PC1'] = coords[:, 0]
df['PC2'] = coords[:, 1]
# print("df :",df)

plt.figure(figsize=(8, 6))
# Plot Noise (-1) in gray
sns.scatterplot(data=df[df.dbscan_label == -1], x='PC1', y='PC2', color='lightgray', s=15, label='Noise')
# Plot Clusters
sns.scatterplot(data=df[df.dbscan_label >= 0], x='PC1', y='PC2', hue='dbscan_label', palette='tab10', s=25)

plt.title(f"AFCluster Landscape: {jobname}")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
import os
from pathlib import Path
import colabfold.batch as cf_batch

# Check what default_data_dir colabfold uses
print("=== ColabFold default_data_dir ===")
print(cf_batch.default_data_dir)

# Download monomer ptm params. These modules will be need to call AF2 model
print("\n=== Downloading AlphaFold monomer params ===")
print("This downloads params_model_1_ptm through model_5_ptm")
print("File size: ~3.5GB — this will take 5-10 minutes on Colab\n")

try:
    cf_batch.download_alphafold_params(
        model_type = "alphafold2_ptm",
        data_dir   = cf_batch.default_data_dir
    )
    print("\n✅ Download complete")

    # Confirm files are present
    params_dir = Path(cf_batch.default_data_dir) / "params"
    npz_files  = sorted([f for f in os.listdir(params_dir)
                         if f.endswith(".npz")])
    print(f"✅ Params dir: {params_dir}")
    print(f"✅ Files found: {npz_files}")

except Exception as e:
    print(f"❌ Download failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================================================
# CELL 9 — DROPOUT CONFIG
# ----------------------------------------------------------------------------
# This cell only sets configuration variables — no computation runs here.
# All variables are read by Cell 10 (Module 4).
# ============================================================================

# ----------------------------------------------------------------------------
# CONTROL 1: How many cluster sub-MSAs to use from Module 3 output.
#
# Module 3 saves files to cluster_output_msas/cluster_0.a3m, cluster_1.a3m...
# Set to an integer to use only the first N clusters.
# Set to None to use ALL clusters Module 3 produced.
# Recommended: start with 3–5 for a test run, then use None for full research.
# ----------------------------------------------------------------------------
N_CLUSTERS_TO_USE = 10        # int or None

# ----------------------------------------------------------------------------
# CONTROL 2: Number of stochastic ensemble structures per sub-MSA.
#
# Each ensemble member = one full AlphaFold forward pass with:
#   - dropout ACTIVE  (eval_dropout = True)
#   - freshly masked MSA columns
#   - unique random seed
# This is equivalent to AFSample2's --nstruct parameter.
# Recommended: 3 for testing, 10–20 for research.
# ----------------------------------------------------------------------------
N_ENSEMBLES_PER_CLUSTER = 1  # int, minimum 1

# ----------------------------------------------------------------------------
# CONTROL 3: MSA column masking fraction (AFSample2 --msa_rand_fraction).
#
# Fraction of aligned MSA columns replaced with residue index 20 (X=unknown).
# Applied to homologue rows only. Query row (row 0) is NEVER masked.
# AFSample2 experimentally validated range: 0.0 – 0.30
#   0.00 = pure dropout diversity only (fastest)
#   0.15 = AFSample2 default          (recommended)
#   0.30 = aggressive perturbation    (more diversity, lower mean pLDDT)
# ----------------------------------------------------------------------------
MSA_RAND_FRACTION = 0.15      # float, 0.0 to 0.30

# ----------------------------------------------------------------------------
# CONTROL 4: Base random seed.
# Each ensemble member i gets seed = DROPOUT_SEED_BASE + i
# Ensures reproducibility while giving each forward pass a unique seed.
# ----------------------------------------------------------------------------
DROPOUT_SEED_BASE = 42        # int

# ----------------------------------------------------------------------------
# CONTROL 5: AlphaFold params directory: /root/.cache/colabfold/params
# Only change this if see a params loading error in Module 4.
# ----------------------------------------------------------------------------
AF_DATA_DIR = "/root/.cache/colabfold"   # parent of the params/ folder

# ── Print summary ─────────────────────────────────────────────────────────
import os
clusters_dir = clusters_dir
if os.path.exists(clusters_dir):
    all_files   = sorted([f for f in os.listdir(clusters_dir)
                          if f.endswith(".a3m")])
    n_available = len(all_files)
else:
    n_available = 0

n_to_use = min(N_CLUSTERS_TO_USE, n_available) \
           if N_CLUSTERS_TO_USE is not None else n_available

print("=" * 60)
print("DROPOUT CONFIG SUMMARY")
print("=" * 60)
print(f"  Clusters available      : {n_available}")
print(f"  N_CLUSTERS_TO_USE       : {N_CLUSTERS_TO_USE}"
      f"  →  will use {n_to_use}")
print(f"  N_ENSEMBLES_PER_CLUSTER : {N_ENSEMBLES_PER_CLUSTER}")
print(f"  MSA_RAND_FRACTION       : {MSA_RAND_FRACTION}")
print(f"  DROPOUT_SEED_BASE       : {DROPOUT_SEED_BASE}")
print(f"  AF_DATA_DIR             : {AF_DATA_DIR}")
print(f"  AlphaFold models        : 5  (model_1 – model_5, alphafold2_ptm)")
print("-" * 60)
print(f"  Total forward passes    : "
      f"{n_to_use} clusters × "
      f"{N_ENSEMBLES_PER_CLUSTER} ensembles × 5 models"
      f" = {n_to_use * N_ENSEMBLES_PER_CLUSTER * 5}")
print("=" * 60)

if n_available == 0:
    print("\n⚠️  WARNING: cluster_output_msas/ is empty or missing.")
    print("   Run Module 3 before running Module 4.")

In [ ]:
# ============================================================================
# CELL 10 — MODULE 4: STOCHASTIC DROPOUT INFERENCE
# ----------------------------------------------------------------------------
# Prerequisites:
#   Cell 1  → jobname, query_sequence
#   Cell 9  → N_CLUSTERS_TO_USE, N_ENSEMBLES_PER_CLUSTER,
#              MSA_RAND_FRACTION, DROPOUT_SEED_BASE, AF_DATA_DIR
#   Module 3 → cluster_output_msas/cluster_{id}.a3m
#
# Design:
#   - Uses colabfold.batch public API throughout (no alphafold.model.model
#     direct import) — this avoids the broken TF Lite .so files entirely
#   - load_models_and_params() is called with use_dropout=True, which sets
#     model_config.model.global_config.eval_dropout = True internally —
#     this is the exact AFSample2 mechanism
#   - MSA masking is applied to feature_dict['msa'] after generate_input_feature()
#     builds it — identical to AFSample2's masking loop:
#       for col in columns_to_randomize:
#           msa[1:, col] = np.array([20] * (msa.shape[0] - 1))
#   - predict_structure() runs the forward pass and saves PDB + JSON
#     automatically — we do not write files manually
# ============================================================================


# ============================================================================
# SECTION 1: IMPORTS
# All imports here are from colabfold.batch which is confirmed working.
# alphafold.model.model is NOT imported here — it is loaded inside
# load_models_and_params() as a deferred local import, which avoids
# the TF Lite broken .so chain entirely.
# ============================================================================

import os
import copy
import json
import time
import traceback
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# colabfold.batch — confirmed fully importable in your environment
from colabfold.batch import (
    generate_input_feature,   # builds feature_dict from a3m string
    predict_structure,        # runs inference, saves PDB + JSON
    set_model_type,           # resolves "auto" to correct model string
)
# from colabfold.input import unserialize_msa   # converts a3m string to MSA lists
from colabfold.batch import unserialize_msa

print("✅ colabfold.batch imports successful")

# load_models_and_params is imported here as a NAME ONLY reference.
# The actual alphafold.model.model import happens inside the function
# when it is first called — deferred exactly as colabfold.batch.run() does it.
# This is what prevents the TF Lite crash.
from colabfold.alphafold.models import load_models_and_params
print("✅ load_models_and_params imported")


# ============================================================================
# SECTION 2: CONSTANTS
# ============================================================================

# AlphaFold residue encoding (residue_constants.restypes order):
# A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
# 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19
# Index 20 = X (unknown) — AFSample2 uses this for MSA column masking
UNKNOWN_IDX  = 20

# Model type for monomer with pTM head — confirmed present in your params dir:
# params_model_1_ptm.npz ... params_model_5_ptm.npz
MODEL_TYPE   = "alphafold2_ptm"

# All 5 models — same set AFSample2 uses by default for monomer
MODEL_ORDER  = [1, 2, 3, 4, 5]
NUM_MODELS   = 5

# max_seq=512 is ColabFold's default for alphafold2_ptm monomers
MAX_SEQ      = 512


# ============================================================================
# SECTION 3: HELPER — PARSE .a3m FILE
# ============================================================================

def read_a3m_file(a3m_path):
    """
    Read a .a3m file and return its raw string content.
    unserialize_msa() expects a list containing this string.
    """
    with open(a3m_path, "r") as fh:
        return fh.read()


def get_query_sequence_from_a3m(a3m_string):
    """
    Extract the query sequence (first sequence) from an a3m string.
    Discards lowercase insertion characters.
    Returns uppercase aligned sequence string.
    """
    for line in a3m_string.splitlines():
        line = line.strip()
        if not line or line.startswith(">"):
            continue
        # First non-header line = query sequence
        return "".join(c for c in line if c.isupper() or c == "-")
    return ""


# ============================================================================
# SECTION 4: HELPER — MSA COLUMN MASKING (AFSample2 exact logic)
# ============================================================================

def mask_msa_columns(feature_dict, msa_rand_fraction, rng):
    """
    Apply AFSample2's MSA column masking logic to feature_dict['msa'].

    Exact AFSample2 logic from run_afsample2.py:
        columns_to_randomize = np.random.choice(
            range(0, nres),
            size=int(nres * msa_rand_fraction),
            replace=False
        )
        for col in columns_to_randomize:
            msa[1:, col] = np.array([20] * (msa.shape[0] - 1))

    Rules:
      - Row 0 (query) is NEVER modified
      - Rows 1+ (homologues) are masked with index 20 (X = unknown)
      - Column selection is without replacement

    Parameters
    ----------
    feature_dict      : dict   output of generate_input_feature()
    msa_rand_fraction : float  fraction of columns to mask
    rng               : np.random.Generator  seeded for reproducibility

    Returns
    -------
    masked_dict  : dict  deep copy of feature_dict with masked msa
    masked_cols  : list  column indices that were masked (for logging)
    """
    masked_dict = copy.deepcopy(feature_dict)

    if msa_rand_fraction <= 0.0:
        return masked_dict, []

    msa       = masked_dict["msa"]
    n_seqs    = msa.shape[0]
    query_len = msa.shape[1]

    if n_seqs < 2:
        # Only query present — nothing to mask
        return masked_dict, []

    # AFSample2 exact column selection
    n_cols_to_mask = int(query_len * msa_rand_fraction)
    if n_cols_to_mask == 0:
        return masked_dict, []

    masked_cols = rng.choice(
        range(0, query_len),
        size=n_cols_to_mask,
        replace=False
    ).tolist()

    # AFSample2 exact masking — rows 1+ only, index 20 (X)
    for col in masked_cols:
        msa[1:, col] = np.array([UNKNOWN_IDX] * (n_seqs - 1))

    masked_dict["msa"] = msa
    return masked_dict, masked_cols


# ============================================================================
# SECTION 5: LOAD MODEL RUNNERS WITH use_dropout=True
# ============================================================================
# load_models_and_params() is called once and reused for all clusters
# and all ensemble members. This avoids recompiling JAX kernels.
#
# use_dropout=True is the single parameter that sets:
#   model_config.model.global_config.eval_dropout = True
# on every runner — confirmed in the models.py source.

print("\n" + "=" * 60)
print("MODULE 4: STOCHASTIC DROPOUT INFERENCE")
print("=" * 60)
print(f"\n── Loading AlphaFold model runners ──────────────────────────")
print(f"   model_type    = {MODEL_TYPE}")
print(f"   model_order   = {MODEL_ORDER}")
print(f"   use_dropout   = True  ← eval_dropout=True on all runners")
print(f"   AF_DATA_DIR   = {AF_DATA_DIR}\n")

model_runner_and_params = load_models_and_params(
    num_models    = NUM_MODELS,
    use_templates = False,          # matches your template_mode='none'
    num_recycles  = 3,              # matches ColabFold default
    model_order   = MODEL_ORDER,
    model_type    = MODEL_TYPE,
    data_dir      = Path(AF_DATA_DIR),
    use_dropout   = True,           # ← THE AFSAMPLE2 KEY FLAG
    max_seq       = MAX_SEQ,
    use_fuse      = True,
    use_bfloat16  = True,
    save_all      = False,
)

print(f"\n✅ {len(model_runner_and_params)} model runners loaded with "
      f"eval_dropout=True")
for model_name, runner, params in model_runner_and_params:
    dropout_status = runner.config.model.global_config.eval_dropout
    print(f"   {model_name}  eval_dropout={dropout_status}")


# # ============================================================================
# # SECTION 6: DISCOVER CLUSTER FILES FROM MODULE 3
# # ============================================================================

# clusters_dir = clusters_dir

# if not os.path.exists(clusters_dir):
#     raise FileNotFoundError(
#         f"'{clusters_dir}' not found. Run Module 3 first."
#     )

# all_cluster_files = sorted([
#     f for f in os.listdir(clusters_dir) if f.endswith(".a3m")
# ])

# if not all_cluster_files:
#     raise FileNotFoundError(
#         f"No .a3m files in '{clusters_dir}'. Run Module 3 first."
#     )

# # Apply N_CLUSTERS_TO_USE limit from Cell 9
# selected_files = (all_cluster_files[:N_CLUSTERS_TO_USE]
#                   if N_CLUSTERS_TO_USE is not None
#                   else all_cluster_files)

# print(f"\n── Cluster sub-MSAs ──────────────────────────────────────────")
# print(f"   Available : {len(all_cluster_files)}")
# print(f"   Selected  : {len(selected_files)}")
# total_runs = len(selected_files) * N_ENSEMBLES_PER_CLUSTER
# print(f"   Total predict_structure calls : "
#       f"{len(selected_files)} × {N_ENSEMBLES_PER_CLUSTER} = {total_runs}")
# print(f"   (Each call runs all {NUM_MODELS} models internally)\n")


# ============================================================================
# SECTION 6: DISCOVER CLUSTER FILES (RANDOM SELECTION WITH CLUSTER 0 INCLUDED)
# ============================================================================
import random

clusters_dir = clusters_dir

if not os.path.exists(clusters_dir):
    raise FileNotFoundError(f"'{clusters_dir}' not found. Run Module 3 first.")

# 1. Get all .a3m files
all_files = [f for f in os.listdir(clusters_dir) if f.endswith(".a3m")]

if not all_files:
    raise FileNotFoundError(f"No .a3m files in '{clusters_dir}'.")

# 2. Identify the Main Cluster (Cluster 0)
main_cluster = "cluster_0.a3m"
if main_cluster not in all_files:
    # Fallback in case cluster_0 isn't named exactly that
    print(f"⚠️ Warning: {main_cluster} not found. Proceeding with random selection only.")
    main_cluster = None

# 3. Create the list of "Other" clusters
other_clusters = [f for f in all_files if f != main_cluster]

# 4. Shuffle the "Other" clusters randomly
# We use DROPOUT_SEED_BASE to ensure your "random" choice is reproducible
rng_selection = random.Random(DROPOUT_SEED_BASE)
rng_selection.shuffle(other_clusters)

# 5. Build the final selection
# We always put the main cluster at index 0
if main_cluster:
    selected_files = [main_cluster]
else:
    selected_files = []

# 6. Fill the rest of the list up to N_CLUSTERS_TO_USE
if N_CLUSTERS_TO_USE is not None:
    # Calculate how many more we need
    needed = N_CLUSTERS_TO_USE - len(selected_files)
    selected_files.extend(other_clusters[:needed])
else:
    # If N_CLUSTERS_TO_USE is None, take everything
    selected_files.extend(other_clusters)

print(f"\n── Cluster sub-MSAs (Random Selection + Cluster 0) ──────────")
print(f"   Available in folder : {len(all_files)}")
print(f"   Selection limit     : {N_CLUSTERS_TO_USE}")
print(f"   Actual Selected     : {len(selected_files)}")
print(f"   Order of execution  : {selected_files}")

total_runs = len(selected_files) * N_ENSEMBLES_PER_CLUSTER
print(f"\n   Total predict_structure calls : "
      f"{len(selected_files)} × {N_ENSEMBLES_PER_CLUSTER} = {total_runs}")

# ============================================================================
# SECTION 7: MAIN INFERENCE LOOP
# ============================================================================

dropout_dir = os.path.join("/content/drive/MyDrive/Research/dropout_predictions", jobname)
os.makedirs(dropout_dir, exist_ok=True)

all_meta = []   # metadata for every completed prediction

for cluster_file in selected_files:

    cluster_id  = cluster_file.replace("cluster_", "").replace(".a3m", "")
    a3m_path    = os.path.join(clusters_dir, cluster_file)
    cluster_out = Path(dropout_dir) / f"cluster_{cluster_id}"
    cluster_out.mkdir(parents=True, exist_ok=True)

    print(f"\n{'═' * 60}")
    print(f"  CLUSTER {cluster_id}  —  {cluster_file}")
    print(f"{'═' * 60}")

    # ── Step 1: Read .a3m file ────────────────────────────────────────────
    a3m_string = read_a3m_file(a3m_path)
    cluster_query_seq = get_query_sequence_from_a3m(a3m_string)
    query_len = len(cluster_query_seq)
    print(f"  Query length : {query_len}")

    # ── Step 2: Build base feature dict via ColabFold API ─────────────────
    # unserialize_msa converts the a3m string into the unpaired_msa format
    # that generate_input_feature() expects — same path as colabfold.batch.run()
    try:
        (unpaired_msa,
         paired_msa,
         query_seqs_unique,
         query_seqs_cardinality,
         template_features) = unserialize_msa(
            [a3m_string],           # list of a3m strings
            cluster_query_seq,      # query sequence string
        )

        base_feature_dict, _ = generate_input_feature(
            query_seqs_unique      = query_seqs_unique,
            query_seqs_cardinality = query_seqs_cardinality,
            unpaired_msa           = unpaired_msa,
            paired_msa             = paired_msa,
            template_features      = template_features,
            is_complex             = False,
            model_type             = MODEL_TYPE,
            max_seq                = MAX_SEQ,
        )
    except Exception as e:
        print(f"  ❌ Feature generation failed for cluster {cluster_id}: {e}")
        traceback.print_exc()
        continue

    n_seqs = base_feature_dict["msa"].shape[0]
    print(f"  Sequences in sub-MSA : {n_seqs}")
    print(f"  feature_dict['msa'] shape : {base_feature_dict['msa'].shape}")

    cluster_meta = []

    # ── Step 3: Ensemble loop ──────────────────────────────────────────────
    for ensemble_idx in range(N_ENSEMBLES_PER_CLUSTER):

        # Seed formula mirrors AFSample2:
        # model_random_seed = model_index + random_seed * num_models
        ensemble_seed = ensemble_idx + DROPOUT_SEED_BASE * NUM_MODELS
        rng           = np.random.default_rng(ensemble_seed)

        print(f"\n  ── Ensemble {ensemble_idx + 1}/{N_ENSEMBLES_PER_CLUSTER}"
              f"  seed={ensemble_seed} ──")

        # ── Step 4: Apply AFSample2 MSA column masking ───────────────────
        # Deep copy base_feature_dict then mask feature_dict['msa']
        # This is identical to AFSample2's:
        #   for col in columns_to_randomize:
        #       msa[1:, col] = np.array([20] * (msa.shape[0] - 1))
        masked_feature_dict, masked_cols = mask_msa_columns(
            feature_dict      = base_feature_dict,
            msa_rand_fraction = MSA_RAND_FRACTION,
            rng               = rng,
        )
        print(f"     MSA columns masked : {len(masked_cols)} / {query_len}")

        # ── Step 5: Run predict_structure ────────────────────────────────
        # predict_structure() internally calls:
        #   model_runner.process_features(feature_dict, random_seed=seed)
        #   model_runner.predict(input_features, random_seed=seed)
        # Because eval_dropout=True on every runner (set in Section 5),
        # dropout is active during predict() — each call samples different
        # dropout masks in the Evoformer and structure module.
        # predict_structure saves PDB and JSON files automatically.

        prefix = (f"{jobname}"
                  f"_cluster{cluster_id}"
                  f"_ensemble{ensemble_idx:03d}"
                  f"_seed{ensemble_seed}"
                  f"_msa{MSA_RAND_FRACTION:.2f}"
                  f"_dropout")

        t0 = time.time()
        try:
            results = predict_structure(
                prefix                  = prefix,
                result_dir              = cluster_out,
                feature_dict            = masked_feature_dict,
                is_complex              = False,
                use_templates           = False,
                sequences_lengths       = [query_len],
                pad_len                 = query_len,
                model_type              = MODEL_TYPE,
                model_runner_and_params = model_runner_and_params,
                num_relax               = 0,
                rank_by                 = "plddt",
                random_seed             = ensemble_seed,
                num_seeds               = 1,
                stop_at_score           = 100,
                save_all                = False,
                save_recycles           = False,
            )
        except Exception as e:
            print(f"     ❌ predict_structure failed: {e}")
            traceback.print_exc()
            continue

        elapsed = time.time() - t0

        # ── Step 6: Extract metrics from results ──────────────────────────
        # predict_structure returns:
        # {"rank": [rank_strings], "metric": [metric_dicts], "result_files": [...]}
        # metric dicts contain mean_plddt, ptm — sorted best-first
        best_metric = results["metric"][0] if results["metric"] else {}
        mean_plddt  = best_metric.get("mean_plddt", float("nan"))
        ptm         = best_metric.get("ptm",        float("nan"))
        best_rank   = results["rank"][0] if results["rank"] else "unknown"

        print(f"     ✅ best rank  : {best_rank}")
        print(f"        pLDDT     : {mean_plddt:.2f}")
        print(f"        pTM       : {ptm:.3f}")
        print(f"        time      : {elapsed:.1f}s")
        print(f"        files     : {len(results['result_files'])} saved")

        meta = {
            "cluster_id"    : cluster_id,
            "ensemble_idx"  : ensemble_idx,
            "seed"          : ensemble_seed,
            "mean_plddt"    : round(float(mean_plddt), 3),
            "ptm"           : round(float(ptm),        3),
            "n_masked_cols" : len(masked_cols),
            "best_rank"     : best_rank,
            "elapsed_s"     : round(elapsed, 2),
            "result_files"  : [str(f) for f in results["result_files"]],
        }
        cluster_meta.append(meta)
        all_meta.append(meta)

    # ── Per-cluster summary ────────────────────────────────────────────────
    if cluster_meta:
        plddts = [m["mean_plddt"] for m in cluster_meta
                  if not np.isnan(m["mean_plddt"])]
        if plddts:
            print(f"\n  Cluster {cluster_id} summary:")
            print(f"  Completed : {len(plddts)} / {N_ENSEMBLES_PER_CLUSTER}")
            print(f"  pLDDT     : mean={np.mean(plddts):.2f}"
                  f"  std={np.std(plddts):.2f}"
                  f"  best={max(plddts):.2f}"
                  f"  worst={min(plddts):.2f}")


# ============================================================================
# SECTION 8: SAVE METADATA + SUMMARY PLOT
# ============================================================================

meta_path = os.path.join(dropout_dir, "all_predictions_meta.json")
with open(meta_path, "w") as fh:
    json.dump(all_meta, fh, indent=2)
print(f"\n✅ Metadata saved → {meta_path}")

# Summary boxplot: pLDDT per cluster
if all_meta:
    cluster_ids = sorted(set(m["cluster_id"] for m in all_meta))
    data_per_cluster = [
        [m["mean_plddt"] for m in all_meta
         if m["cluster_id"] == cid and not np.isnan(m["mean_plddt"])]
        for cid in cluster_ids
    ]
    valid = [(c, d) for c, d in zip(cluster_ids, data_per_cluster) if d]

    if valid:
        cids, vals = zip(*valid)
        fig, ax = plt.subplots(figsize=(max(6, len(cids) * 1.5), 5))
        ax.boxplot(
            vals,
            labels       = [f"C{c}" for c in cids],
            patch_artist = True,
            boxprops     = dict(facecolor="steelblue", alpha=0.6),
            medianprops  = dict(color="black", linewidth=2),
            flierprops   = dict(marker="o", markersize=3, alpha=0.5),
        )
        ax.axhline(70, color="green",  linestyle="--", alpha=0.5,
                   label="pLDDT=70 (confident)")
        ax.axhline(50, color="orange", linestyle="--", alpha=0.5,
                   label="pLDDT=50 (low confidence)")
        ax.set_xlabel(
            "Cluster  (evolutionary sub-population — DBSCAN Module 2)")
        ax.set_ylabel("Mean pLDDT")
        ax.set_ylim(0, 100)
        ax.set_title(
            f"Stochastic dropout ensemble — {jobname}\n"
            f"use_dropout=True  |  "
            f"MSA masking={MSA_RAND_FRACTION}  |  "
            f"{N_ENSEMBLES_PER_CLUSTER} ensembles per cluster"
        )
        ax.legend()
        plt.tight_layout()
        plt.show()

print(f"\n{'=' * 60}")
print(f"MODULE 4 COMPLETE")
print(f"{'=' * 60}")
print(f"  PDB files  → {dropout_dir}/cluster_*/")
print(f"  Metadata   → {meta_path}")
print(f"  Completed  : {len(all_meta)} / "
      f"{len(selected_files) * N_ENSEMBLES_PER_CLUSTER} ensemble runs")

In [ ]:
import py3Dmol
import os
import glob
from ipywidgets import interact, Dropdown
from IPython.display import display, clear_output

def get_all_pdb_files(base_path):
    return sorted(glob.glob(os.path.join(base_path, "**/*.pdb"), recursive=True))

def visualize_pdb(pdb_path):
    # 1. Clear the output area where the previous viewer was
    clear_output(wait=True)

    if not os.path.exists(pdb_path):
        print(f"File not found: {pdb_path}")
        return

    with open(pdb_path, 'r') as f:
        pdb_str = f.read()

    # 2. Re-create the viewer
    view = py3Dmol.view(width=800, height=400)
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()

    print(f"Viewing: {os.path.basename(pdb_path)}")

    # 3. Explicitly display the new view
    view.show()

# --- Usage ---
prediction_folder = dropout_dir
pdb_files = get_all_pdb_files(prediction_folder)

if not pdb_files:
    print("No PDB files found! Check your path.")
else:
    # Use the dropdown interaction
    interact(visualize_pdb, pdb_path=Dropdown(options=pdb_files, description='Select PDB:'));

In [ ]:
import requests
import time
import json

sequence = query_sequence

# Submit BLAST search against PDB
print("Submitting BLAST search against PDB...")
blast_url = "https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi"

params = {
    "CMD"         : "Put",
    "PROGRAM"     : "blastp",
    "DATABASE"    : "pdb",
    "QUERY"       : sequence,
    "FORMAT_TYPE" : "JSON2",
    "HITLIST_SIZE": "50",
    "EXPECT"      : "10",
}
response = requests.post(blast_url, data=params)
rid = response.text.split("RID = ")[1].split("\n")[0].strip()
print(f"BLAST RID: {rid}")
print("Waiting for results (60 seconds)...")

time.sleep(60)

# Retrieve results
params_get = {
    "CMD"         : "Get",
    "RID"         : rid,
    "FORMAT_TYPE" : "JSON2",
}
result = requests.get(blast_url, params=params_get)

# Parse PDB hits
lines = result.text.split("\n")
print("\n=== Top PDB hits ===")
for line in lines:
    if "pdb" in line.lower() and "|" in line:
        print(line.strip())

In [ ]:
import requests

def download_pdb(pdb_id, output_path):
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    print(f"Downloading {pdb_id.upper()} from RCSB...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(output_path, "w") as f:
            f.write(response.text)
        print(f"✅ Saved to {output_path}")
    else:
        print(f"❌ Failed to download {pdb_id}. Status code: {response.status_code}")

# Download experimental open and closed states for your Tier 2 evaluation
download_pdb("4ake", "4ake.pdb")  # Open State
download_pdb("2eck", "2eck.pdb")  # Closed State

In [ ]:
import sys
import subprocess

print(f"Current active python: {sys.executable}")
print("Installing MDAnalysis directly to this interpreter...")

# This forces pip to target the exact running kernel environment
subprocess.run([sys.executable, "-m", "pip", "install", "MDAnalysis", "-q", "--break-system-packages"])

# Invalidate cache to force Python to re-scan modules
import importlib
importlib.invalidate_caches()

import MDAnalysis as mda
print("✅ Success! MDAnalysis imported successfully.")

# MODULE 5: ENSEMBLE EVALUATION

In [ ]:
job_name

In [ ]:

# Computes all Tier 1 metrics from your predicted PDB files.
# Tier 2 metrics activate automatically if you provide experimental PDB paths.

# Dependencies: MDAnalysis (structure loading + RMSD), scikit-learn (PCA)
# Install: !pip install MDAnalysis -q


# ── Install MDAnalysis if not present ────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "MDAnalysis", "-q",
                "--break-system-packages"], capture_output=True)

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

print("✅ MDAnalysis imported successfully")


# ============================================================================
# CONFIGURATION
# ============================================================================

# Directory containing your Module 4 predictions
# DROPOUT_DIR   = f"dropout_predictions/{jobname}"
DROPOUT_DIR = "/content/drive/MyDrive/Research/dropout_predictions/Test1_by_channa_27f14_1"

# ── Tier 2 (optional): paths to experimental PDB files ───────────────────
# After running BLAST and downloading PDB structures, add their paths here.
# Leave as empty list [] if you have no experimental structures yet.
# Example: EXPERIMENTAL_PDBS = ["/content/4ake.pdb", "/content/1ake.pdb"]
EXPERIMENTAL_PDBS = ["4ake.pdb", "2eck.pdb"]

# RMSD threshold for coverage calculation (Angstroms)
RMSD_THRESHOLD = 2.0

# Atoms to use for RMSD — Cα only, standard for ensemble comparison
ATOM_SELECTION = "backbone and name CA"


# ============================================================================
# SECTION 1: COLLECT ALL PREDICTED PDB FILES
# ============================================================================

print("\n── Collecting predicted PDB files ───────────────────────────")

all_pdb_files   = []
cluster_pdb_map = {}   # {cluster_id: [pdb_path, ...]}

for cluster_dir in sorted(Path(DROPOUT_DIR).glob("cluster_*")):
    if not cluster_dir.is_dir():
        continue
    cluster_id = cluster_dir.name.replace("cluster_", "")
    # Collect only rank_001 files (best prediction per ensemble call)
    # These are the highest-confidence structures from each forward pass
    pdbs = sorted(cluster_dir.glob("*rank_001*.pdb"))
    if not pdbs:
        # Fallback: collect all unrelaxed PDB files
        pdbs = sorted(cluster_dir.glob("*unrelaxed*.pdb"))
    cluster_pdb_map[cluster_id] = [str(p) for p in pdbs]
    all_pdb_files.extend([str(p) for p in pdbs])

print(f"   Total PDB files collected : {len(all_pdb_files)}")
for cid, pdbs in cluster_pdb_map.items():
    print(f"   Cluster {cid:>3}             : {len(pdbs)} structures")

if not all_pdb_files:
    raise FileNotFoundError(
        f"No PDB files found in {DROPOUT_DIR}. Run Module 4 first."
    )


# ============================================================================
# SECTION 2: LOAD STRUCTURES AND EXTRACT Cα COORDINATES
# ============================================================================

print("\n── Loading structures and extracting Cα coordinates ─────────")

def load_ca_coords(pdb_path, selection=ATOM_SELECTION):
    """
    Load a PDB file and return Cα coordinate array.
    Returns shape (n_residues, 3).
    """
    u      = mda.Universe(pdb_path)
    ca     = u.select_atoms(selection)
    coords = ca.positions.copy()   # shape (n_ca, 3)
    return coords

# Load all predicted structures
# Use first structure as reference for alignment
ref_coords   = None
all_coords   = []   # list of (n_residues, 3) arrays
valid_pdbs   = []   # paths of successfully loaded PDBs
cluster_ids  = []   # cluster id for each structure

for cluster_id, pdb_list in sorted(cluster_pdb_map.items()):
    for pdb_path in pdb_list:
        try:
            coords = load_ca_coords(pdb_path)
            if ref_coords is None:
                ref_coords = coords.copy()
                n_residues = coords.shape[0]
                print(f"   Reference structure : {Path(pdb_path).name}")
                print(f"   n_residues (Cα)     : {n_residues}")
            if coords.shape[0] == n_residues:
                all_coords.append(coords)
                valid_pdbs.append(pdb_path)
                cluster_ids.append(cluster_id)
        except Exception as e:
            print(f"   ⚠️  Could not load {Path(pdb_path).name}: {e}")

print(f"\n✅ Loaded {len(all_coords)} structures successfully")
all_coords  = np.array(all_coords)   # shape (N_structs, n_residues, 3)
cluster_ids = np.array(cluster_ids)


# ============================================================================
# SECTION 3: ALIGN ALL STRUCTURES TO REFERENCE (Cα superposition)
# ============================================================================

print("\n── Aligning structures to reference ─────────────────────────")

def align_to_reference(coords_array, ref):
    """
    Superimpose each structure onto ref using Kabsch algorithm.
    coords_array: (N, L, 3)
    ref         : (L, 3)
    Returns aligned coords_array (N, L, 3)
    """
    aligned = np.zeros_like(coords_array)
    ref_centered = ref - ref.mean(axis=0)

    for i, coords in enumerate(coords_array):
        mob_centered = coords - coords.mean(axis=0)
        # Kabsch algorithm
        H   = mob_centered.T @ ref_centered
        U, S, Vt = np.linalg.svd(H)
        d   = np.linalg.det(Vt.T @ U.T)
        D   = np.diag([1, 1, d])
        R   = Vt.T @ D @ U.T
        aligned[i] = (mob_centered @ R.T) + ref.mean(axis=0)

    return aligned

aligned_coords = align_to_reference(all_coords, ref_coords)
print(f"✅ Aligned {len(aligned_coords)} structures")


# ============================================================================
# SECTION 4: PAIRWISE Cα RMSD MATRIX
# ----------------------------------------------------------------------------
# Primary diversity metric used by AFSample2, AFCluster, AlphaFlow
# Reference: Wayment-Steele et al., Nature 2024; Ströder et al., PLOS CB 2024
# ============================================================================

print("\n── Computing pairwise Cα RMSD matrix ────────────────────────")

n_structs    = len(aligned_coords)
rmsd_matrix  = np.zeros((n_structs, n_structs))

for i in range(n_structs):
    for j in range(i + 1, n_structs):
        diff = aligned_coords[i] - aligned_coords[j]
        rmsd = np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))
        rmsd_matrix[i, j] = rmsd
        rmsd_matrix[j, i] = rmsd

# Extract upper triangle (unique pairs)
upper_tri   = rmsd_matrix[np.triu_indices(n_structs, k=1)]

print(f"✅ Pairwise RMSD computed ({n_structs}×{n_structs} matrix)")
print(f"\n── DIVERSITY METRICS ──────────────────────────────────────────")
print(f"   Mean pairwise Cα RMSD   : {upper_tri.mean():.3f} Å")
print(f"   Std  pairwise Cα RMSD   : {upper_tri.std():.3f} Å")
print(f"   Max  pairwise Cα RMSD   : {upper_tri.max():.3f} Å")
print(f"   Min  pairwise Cα RMSD   : {upper_tri.min():.3f} Å")
print(f"   Median pairwise Cα RMSD : {np.median(upper_tri):.3f} Å")


# ============================================================================
# SECTION 5: PCA OF Cα COORDINATES (conformational landscape)
# ----------------------------------------------------------------------------
# Visualises the conformational space explored by the ensemble.
# Reference: Jing et al., ICML 2024 (AlphaFlow)
# ============================================================================

print("\n── PCA of Cα coordinates ────────────────────────────────────")

# Flatten coordinates: (N_structs, n_residues * 3)
coords_flat = aligned_coords.reshape(n_structs, -1)
coords_scaled = StandardScaler().fit_transform(coords_flat)

pca       = PCA(n_components=2)
pca_coords = pca.fit_transform(coords_scaled)
var_exp   = pca.explained_variance_ratio_ * 100

print(f"   PC1 variance explained : {var_exp[0]:.1f}%")
print(f"   PC2 variance explained : {var_exp[1]:.1f}%")
print(f"   Total                  : {var_exp.sum():.1f}%")


# ============================================================================
# SECTION 6: LOAD SCORES FROM METADATA JSON
# ============================================================================

meta_path = Path(DROPOUT_DIR) / "all_predictions_meta.json"
if meta_path.exists():
    with open(meta_path) as fh:
        all_meta = json.load(fh)
    meta_plddts = [m["mean_plddt"] for m in all_meta]
    meta_ptms   = [m["ptm"]        for m in all_meta]
    meta_cids   = [m["cluster_id"] for m in all_meta]
else:
    meta_plddts, meta_ptms, meta_cids = [], [], []


# ============================================================================
# SECTION 7: TIER 2 — minRMSD TO EXPERIMENTAL STRUCTURES (if available)
# ----------------------------------------------------------------------------
# Reference: Ströder et al., PLOS Comp Biol 2024 (AFSample2 evaluation)
# ============================================================================

exp_coverage = {}   # {exp_pdb_name: min_rmsd}

if EXPERIMENTAL_PDBS:
    print("\n── Tier 2: Coverage of experimental conformations ───────────")
    for exp_pdb in EXPERIMENTAL_PDBS:
        try:
            exp_coords = load_ca_coords(exp_pdb)
            if exp_coords.shape[0] != n_residues:
                print(f"   ⚠️  {Path(exp_pdb).name}: "
                      f"length mismatch ({exp_coords.shape[0]} vs {n_residues})")
                continue

            # Align experimental structure to same reference
            exp_aligned = align_to_reference(
                exp_coords[np.newaxis], ref_coords
            )[0]

            # Compute RMSD from this experimental structure to every ensemble member
            rmsds = []
            for i, pred_coords in enumerate(aligned_coords):
                diff = pred_coords - exp_aligned
                rmsd = np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))
                rmsds.append(rmsd)

            min_rmsd   = min(rmsds)
            best_match = valid_pdbs[np.argmin(rmsds)]
            covered    = min_rmsd <= RMSD_THRESHOLD

            exp_coverage[Path(exp_pdb).name] = {
                "min_rmsd"    : round(min_rmsd, 3),
                "best_match"  : Path(best_match).name,
                "covered"     : covered,
            }
            print(f"   {Path(exp_pdb).name:30s} "
                  f"minRMSD={min_rmsd:.2f}Å  "
                  f"{'✅ COVERED' if covered else '❌ not covered'} "
                  f"(threshold={RMSD_THRESHOLD}Å)")

        except Exception as e:
            print(f"   ❌ Could not process {exp_pdb}: {e}")

    n_covered = sum(v["covered"] for v in exp_coverage.values())
    n_total   = len(exp_coverage)
    if n_total > 0:
        print(f"\n   Coverage rate : {n_covered}/{n_total} "
              f"({n_covered/n_total*100:.1f}%) "
              f"at {RMSD_THRESHOLD}Å threshold")
else:
    print("\n── Tier 2 skipped (no experimental PDBs provided) ──────────")
    print("   Add paths to EXPERIMENTAL_PDBS list to enable coverage analysis.")


# ============================================================================
# SECTION 8: VISUALISATION DASHBOARD
# ============================================================================

print("\n── Generating evaluation dashboard ──────────────────────────")

# Assign colours per cluster
unique_cids    = sorted(set(cluster_ids))
cmap           = plt.cm.get_cmap("tab10", len(unique_cids))
cid_colour_map = {cid: cmap(i) for i, cid in enumerate(unique_cids)}
colours        = [cid_colour_map[cid] for cid in cluster_ids]

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Plot 1: Pairwise RMSD heatmap ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
im  = ax1.imshow(rmsd_matrix, cmap="viridis", aspect="auto")
plt.colorbar(im, ax=ax1, label="Cα RMSD (Å)")
ax1.set_title("Pairwise Cα RMSD matrix\n(all ensemble members)",
              fontsize=10)
ax1.set_xlabel("Structure index")
ax1.set_ylabel("Structure index")

# Add cluster boundary lines
cluster_boundaries = []
current = cluster_ids[0]
for idx, cid in enumerate(cluster_ids):
    if cid != current:
        cluster_boundaries.append(idx - 0.5)
        current = cid
for b in cluster_boundaries:
    ax1.axhline(b, color="red", linewidth=0.8, alpha=0.7)
    ax1.axvline(b, color="red", linewidth=0.8, alpha=0.7)

# ── Plot 2: Pairwise RMSD distribution ───────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(upper_tri, bins=30, color="steelblue", edgecolor="white",
         alpha=0.8)
ax2.axvline(upper_tri.mean(), color="red", linestyle="--",
            label=f"Mean={upper_tri.mean():.2f}Å")
ax2.axvline(upper_tri.max(),  color="orange", linestyle="--",
            label=f"Max={upper_tri.max():.2f}Å")
ax2.set_xlabel("Cα RMSD (Å)")
ax2.set_ylabel("Count")
ax2.set_title("Pairwise RMSD distribution\n(ensemble diversity)",
              fontsize=10)
ax2.legend(fontsize=8)

# ── Plot 3: PCA conformational landscape ─────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
for cid in unique_cids:
    mask = cluster_ids == cid
    ax3.scatter(
        pca_coords[mask, 0], pca_coords[mask, 1],
        c=[cid_colour_map[cid]], label=f"Cluster {cid}",
        s=60, alpha=0.8, edgecolors="white", linewidths=0.5
    )
# Mark reference structure (index 0)
ax3.scatter(pca_coords[0, 0], pca_coords[0, 1],
            marker="*", s=250, c="red", zorder=5, label="Reference")
ax3.set_xlabel(f"PC1 ({var_exp[0]:.1f}%)")
ax3.set_ylabel(f"PC2 ({var_exp[1]:.1f}%)")
ax3.set_title("Conformational landscape (PCA)\ncoloured by cluster",
              fontsize=10)
ax3.legend(fontsize=7, loc="best")

# ── Plot 4: pLDDT distribution per cluster ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
if meta_plddts:
    unique_meta_cids = sorted(set(meta_cids))
    data_by_cluster  = [
        [meta_plddts[i] for i, c in enumerate(meta_cids) if c == cid]
        for cid in unique_meta_cids
    ]
    bp = ax4.boxplot(
        data_by_cluster,
        tick_labels    = [f"C{c}" for c in unique_meta_cids],
        patch_artist   = True,
        boxprops       = dict(facecolor="steelblue", alpha=0.6),
        medianprops    = dict(color="black", linewidth=2),
    )
    ax4.axhline(90, color="green",  linestyle="--", alpha=0.6,
                label="pLDDT=90 (very high)")
    ax4.axhline(70, color="orange", linestyle="--", alpha=0.6,
                label="pLDDT=70 (confident)")
    ax4.set_ylim(0, 100)
    ax4.set_ylabel("Mean pLDDT")
    ax4.set_xlabel("Cluster")
    ax4.set_title("pLDDT per cluster\n(AlphaFold confidence)",
                  fontsize=10)
    ax4.legend(fontsize=7)

# ── Plot 5: pTM distribution per cluster ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
if meta_ptms:
    data_ptm_by_cluster = [
        [meta_ptms[i] for i, c in enumerate(meta_cids) if c == cid]
        for cid in unique_meta_cids
    ]
    ax5.boxplot(
        data_ptm_by_cluster,
        tick_labels  = [f"C{c}" for c in unique_meta_cids],
        patch_artist = True,
        boxprops     = dict(facecolor="coral", alpha=0.6),
        medianprops  = dict(color="black", linewidth=2),
    )
    ax5.axhline(0.7, color="green",  linestyle="--", alpha=0.6,
                label="pTM=0.70")
    ax5.axhline(0.5, color="orange", linestyle="--", alpha=0.6,
                label="pTM=0.50")
    ax5.set_ylim(0, 1)
    ax5.set_ylabel("pTM score")
    ax5.set_xlabel("Cluster")
    ax5.set_title("pTM per cluster\n(global fold confidence)",
                  fontsize=10)
    ax5.legend(fontsize=7)

# ── Plot 6: Per-structure RMSD to reference ───────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
rmsds_to_ref = [
    np.sqrt(np.mean(np.sum((aligned_coords[i] - ref_coords) ** 2, axis=1)))
    for i in range(n_structs)
]
ax6.bar(range(n_structs), rmsds_to_ref,
        color=colours, edgecolor="white", alpha=0.8)
ax6.set_xlabel("Structure index")
ax6.set_ylabel("Cα RMSD to reference (Å)")
ax6.set_title("Per-structure RMSD to reference\n(coloured by cluster)",
              fontsize=10)
handles = [plt.Rectangle((0,0),1,1, color=cid_colour_map[cid])
           for cid in unique_cids]
ax6.legend(handles, [f"C{c}" for c in unique_cids],
           fontsize=7, loc="upper right")

# ── Plot 7: Cumulative coverage curve (Tier 2, if available) ─────────────
ax7 = fig.add_subplot(gs[2, 0])
if exp_coverage:
    thresholds = np.arange(0, 10.1, 0.1)
    for exp_name, info in exp_coverage.items():
        # For each threshold, check if minRMSD is below it
        coverage = [1.0 if info["min_rmsd"] <= t else 0.0
                    for t in thresholds]
        ax7.plot(thresholds, coverage, label=exp_name, linewidth=2)
    ax7.axvline(RMSD_THRESHOLD, color="gray", linestyle="--",
                alpha=0.7, label=f"Threshold={RMSD_THRESHOLD}Å")
    ax7.set_xlabel("RMSD threshold (Å)")
    ax7.set_ylabel("Covered (0/1)")
    ax7.set_title("Experimental state coverage\nvs RMSD threshold",
                  fontsize=10)
    ax7.legend(fontsize=7)
    ax7.set_ylim(-0.05, 1.05)
else:
    ax7.text(0.5, 0.5,
             "Tier 2 metrics unavailable\n\nAdd experimental PDB paths\nto EXPERIMENTAL_PDBS list",
             ha="center", va="center", fontsize=10,
             transform=ax7.transAxes,
             bbox=dict(boxstyle="round", facecolor="lightyellow",
                       alpha=0.8))
    ax7.set_title("Experimental coverage\n(not yet available)", fontsize=10)
    ax7.axis("off")

# ── Plot 8: Summary statistics table ─────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 1:])
ax8.axis("off")

summary_rows = [
    ["Metric", "Value", "Interpretation"],
    ["Total ensemble size",
     str(n_structs),
     "Number of predicted structures"],
    ["Mean pairwise RMSD",
     f"{upper_tri.mean():.3f} Å",
     ">1Å = meaningful diversity"],
    ["Max pairwise RMSD",
     f"{upper_tri.max():.3f} Å",
     "Total conformational range"],
    ["Mean pLDDT",
     f"{np.mean(meta_plddts):.2f}" if meta_plddts else "N/A",
     ">90=very high, >70=confident"],
    ["Mean pTM",
     f"{np.mean(meta_ptms):.3f}" if meta_ptms else "N/A",
     ">0.7=reliable global fold"],
    ["N clusters",
     str(len(unique_cids)),
     "Evolutionary sub-populations used"],
]

if exp_coverage:
    n_cov = sum(v["covered"] for v in exp_coverage.values())
    summary_rows.append([
        f"Coverage at {RMSD_THRESHOLD}Å",
        f"{n_cov}/{len(exp_coverage)} "
        f"({n_cov/len(exp_coverage)*100:.0f}%)",
        "Fraction of exp. states captured"
    ])

table = ax8.table(
    cellText   = summary_rows[1:],
    colLabels  = summary_rows[0],
    cellLoc    = "center",
    loc        = "center",
    bbox       = [0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2c5f8a")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#f0f4f8")
ax8.set_title("Evaluation summary", fontsize=10, pad=10)

fig.suptitle(
    f"Conformational Ensemble Evaluation — {jobname}\n"
    f"Pipeline: DBSCAN sub-MSA clustering + AFSample2 stochastic dropout",
    fontsize=12, fontweight="bold", y=1.01
)
plt.savefig(f"{DROPOUT_DIR}/evaluation_dashboard.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Dashboard saved → {DROPOUT_DIR}/evaluation_dashboard.png")


# ============================================================================
# SECTION 9: PRINT FINAL EVALUATION REPORT
# ============================================================================

print("\n" + "=" * 60)
print("EVALUATION REPORT")
print("=" * 60)
print(f"  Protein        : {jobname}")
print(f"  Ensemble size  : {n_structs} structures")
print(f"  Clusters used  : {len(unique_cids)}")
print()
print("  DIVERSITY (Tier 1)")
print(f"    Mean pairwise RMSD   : {upper_tri.mean():.3f} Å")
print(f"    Max  pairwise RMSD   : {upper_tri.max():.3f} Å")
print(f"    Std  pairwise RMSD   : {upper_tri.std():.3f} Å")
print()
print("  CONFIDENCE (Tier 1)")
if meta_plddts:
    print(f"    Mean pLDDT          : {np.mean(meta_plddts):.2f}")
    print(f"    Mean pTM            : {np.mean(meta_ptms):.3f}")
print()
if exp_coverage:
    print(f"  COVERAGE (Tier 2, threshold={RMSD_THRESHOLD}Å)")
    for name, info in exp_coverage.items():
        status = "✅ covered" if info["covered"] else "❌ not covered"
        print(f"    {name:30s} minRMSD={info['min_rmsd']:.2f}Å  {status}")
else:
    print("  COVERAGE (Tier 2)")
    print("    Not computed — add experimental PDB paths to EXPERIMENTAL_PDBS")
print("=" * 60)

To do:

1. Understand and recheck the DBSCAN module: DONE
2. Remove old DBSCAN module: DONE
3. Understand DROPOUT logic.
4. Check DROPOUT module.
5. Update the project readme.md
6. Calculate metrics and compare with existing tools.
7. Implementation a structural visulization module: DONE
8. Check the hard coded max_value=108 in OHE in DBSCAN algorithm
9. Verify the evaluation module implementation.
10. create v3: DONE
11. Find what is mmseques2_uniref_env?
12. Do I use clusters which have maximum number of sequences?